In [ ]:
%matplotlib inline

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import schemdraw
import schemdraw.elements as elm
from matplotlib.ticker import NullFormatter
from ipywidgets import interactive, FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, HTMLMath
from IPython.display import display

s, R, L, C = sp.symbols('s R L C', positive=True, real=True)

# ------------------------------------------------------------
# TRANSFER FUNCTIONS
# ------------------------------------------------------------

def get_transfer_function(filter_type):

    if filter_type == 'Low-pass':
        H = 1 / (L*C*s**2 + (L/R)*s + 1)
        wc = 1 / sp.sqrt(L*C)
        Q = R * sp.sqrt(C/L)

    elif filter_type == 'High-pass':
        H = L*C*s**2 / (L*C*s**2 + (L/R)*s + 1)
        wc = 1 / sp.sqrt(L*C)
        Q = R * sp.sqrt(C/L)

    elif filter_type == 'Band-pass':
        H = L*s / (R*L*C*s**2 + L*s + R)
        wc = 1 / sp.sqrt(L*C)
        Q = R * sp.sqrt(C/L)

    elif filter_type == 'Band-stop':
        H = (L*C*s**2 + 1) / (L*C*s**2 + R*C*s + 1)
        wc = 1 / sp.sqrt(L*C)
        Q = sp.sqrt(L/C) / R

    return sp.factor(H), sp.simplify(wc), sp.simplify(Q)

# ------------------------------------------------------------
# CONTROLS
# ------------------------------------------------------------

filter_title = HTML(value="<b>Filter Type:</b>")
filter_radio = RadioButtons(options=['Low-pass', 'High-pass', 'Band-pass', 'Band-stop'], value='Low-pass', description='', layout=Layout(width='150px'))

r_title = HTML(value="<b style='color:red;'>Resistance R (Ω)</b>")
r_slider = FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='', readout=True, readout_format='.2f', continuous_update=True, style={'handle_color': 'red'}, layout=Layout(width='240px'))

l_title = HTML(value="<b style='color:blue;'>Inductance L (H)</b>")
l_slider = FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='', readout=True, readout_format='.2f', continuous_update=True, style={'handle_color': 'blue'}, layout=Layout(width='240px'))

c_title = HTML(value="<b style='color:green;'>Capacitance C (F)</b>")
c_slider = FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description='', readout=True, readout_format='.2f', continuous_update=True, style={'handle_color': 'green'}, layout=Layout(width='240px'))

# ------------------------------------------------------------
# EQUATIONS
# ------------------------------------------------------------

equation_title = HTML(value="<b>Transfer Function and Parameters</b>")
equation_display = HTMLMath(layout=Layout(width='300px'))

# ------------------------------------------------------------
# CIRCUIT
# ------------------------------------------------------------

circuit_title = HTML(value="<b>Circuit</b>")
circuit_display = HTML(layout=Layout(width='380px', height='170px', overflow='hidden'))

def make_circuit_svg(filter_type):

    d = schemdraw.Drawing(show=False, unit=1.6, canvas='svg')

    if filter_type == 'Low-pass':

        d += elm.Dot().at((0, 0))
        d += elm.Line().right().length(0.4)
        d += elm.Inductor().right().length(1.6).label('$L$', loc='top').color('blue')
        d += elm.Line().right().length(1.0)
        d += elm.Dot()
        node = d.here

        d += elm.Line().right().length(0.8)
        d += elm.Dot().label('$V_{out}$', loc='right')

        d += elm.Capacitor().at(node).down().length(2.0).label('$C$', loc='left').color('green')
        bottom_c = d.here

        d += elm.Resistor().at((node[0] + 0.8, node[1])).down().length(2.0).label('$R$', loc='right').color('red')
        bottom_r = d.here

        d += elm.Line().at(bottom_c).right().to(bottom_r)
        d += elm.Line().at((0, -2.0)).right().to(bottom_r)
        d += elm.Dot().at((0, -2.0))
        d += elm.Dot().at(bottom_r)
        d += elm.Label().at((-0.15, -1.0)).label('$V_{in}$')

    elif filter_type == 'High-pass':

        d += elm.Dot().at((0, 0))
        d += elm.Line().right().length(0.4)
        d += elm.Capacitor().right().length(1.6).label('$C$', loc='top').color('green')
        d += elm.Line().right().length(1.0)
        d += elm.Dot()
        node = d.here

        d += elm.Line().right().length(0.8)
        d += elm.Dot().label('$V_{out}$', loc='right')

        d += elm.Inductor().at(node).down().length(2.0).label('$L$', loc='left').color('blue')
        bottom_l = d.here

        d += elm.Resistor().at((node[0] + 0.8, node[1])).down().length(2.0).label('$R$', loc='right').color('red')
        bottom_r = d.here

        d += elm.Line().at(bottom_l).right().to(bottom_r)
        d += elm.Line().at((0, -2.0)).right().to(bottom_r)
        d += elm.Dot().at((0, -2.0))
        d += elm.Dot().at(bottom_r)
        d += elm.Label().at((-0.15, -1.0)).label('$V_{in}$')

    elif filter_type == 'Band-pass':

        d += elm.Dot().at((0, 0))
        d += elm.Line().right().length(0.4)
        d += elm.Resistor().right().length(1.6).label('$R$', loc='top').color('red')
        d += elm.Line().right().length(1.0)
        d += elm.Dot()
        node = d.here

        d += elm.Line().right().length(0.8)
        d += elm.Dot().label('$V_{out}$', loc='right')

        d += elm.Inductor().at(node).down().length(2.0).label('$L$', loc='left').color('blue')
        bottom_l = d.here

        d += elm.Capacitor().at((node[0] + 0.8, node[1])).down().length(2.0).label('$C$', loc='right').color('green')
        bottom_c = d.here

        d += elm.Line().at(bottom_l).right().to(bottom_c)
        d += elm.Line().at((0, -2.0)).right().to(bottom_c)
        d += elm.Dot().at((0, -2.0))
        d += elm.Dot().at(bottom_c)
        d += elm.Label().at((-0.15, -1.0)).label('$V_{in}$')

    elif filter_type == 'Band-stop':

        d += elm.Dot().at((0, 0))
        d += elm.Line().right().length(0.4)
        d += elm.Resistor().right().length(1.6).label('$R$', loc='top').color('red')
        d += elm.Line().right().length(1.0)
        d += elm.Dot()
        node = d.here

        d += elm.Line().right().length(0.8)
        d += elm.Dot().label('$V_{out}$', loc='right')

        d += elm.Inductor().at(node).down().length(0.95).label('$L$', loc='left').color('blue')
        d += elm.Capacitor().down().length(1.05).label('$C$', loc='left').color('green')
        bottom_lc = d.here

        d += elm.Line().at((0, -2.0)).right().length(node[0] + 0.8)
        d += elm.Dot().at((0, -2.0))
        d += elm.Dot().at((node[0] + 0.8, -2.0))
        d += elm.Line().at(bottom_lc).right().to((node[0] + 0.8, -2.0))
        d += elm.Label().at((-0.15, -1.0)).label('$V_{in}$')

    return d.get_imagedata('svg').decode('utf-8')

# ------------------------------------------------------------
# TOP ROW
# ------------------------------------------------------------

filter_box = VBox([filter_title, filter_radio], layout=Layout(width='150px'))
equation_box = VBox([equation_title, equation_display], layout=Layout(width='310px'))
circuit_box = VBox([circuit_title, circuit_display], layout=Layout(width='390px'))

top_row = HBox([filter_box, equation_box, circuit_box], layout=Layout(width='880px', align_items='flex-start', justify_content='space-between'))

# ------------------------------------------------------------
# SLIDERS
# ------------------------------------------------------------

r_box = VBox([r_title, r_slider], layout=Layout(width='250px'))
l_box = VBox([l_title, l_slider], layout=Layout(width='250px'))
c_box = VBox([c_title, c_slider], layout=Layout(width='250px'))

slider_row = HBox([r_box, l_box, c_box], layout=Layout(width='800px', justify_content='space-between', margin='8px 0 12px 0'))

# ------------------------------------------------------------
# INFORMATION MESSAGE
# ------------------------------------------------------------

info_message = HTML(value='', layout=Layout(width='1040px', margin='4px 0 0 0'))

# ------------------------------------------------------------
# MAIN INTERACTIVE FUNCTION
# ------------------------------------------------------------

def plot_filter(filter_type, R_val, L_val, C_val):

    # --------------------------------------------------------
    # TRANSFER FUNCTION AND PARAMETERS
    # --------------------------------------------------------

    H_sym, wc_sym, Q_sym = get_transfer_function(filter_type)

    wc_val = float(wc_sym.subs({R: R_val, L: L_val, C: C_val}))
    Q_val = float(Q_sym.subs({R: R_val, L: L_val, C: C_val}))

    equation_display.value = r"$$\mathcal{H}(s)=" + sp.latex(H_sym) + r"$$" + r"$$\omega_c=" + sp.latex(wc_sym) + rf"={wc_val:.4f}\ \mathrm{{rad/s}}$$" + r"$$Q=" + sp.latex(Q_sym) + rf"={Q_val:.4f}$$"

    # --------------------------------------------------------
    # CIRCUIT
    # --------------------------------------------------------

    circuit_display.value = "<div style='width:100%;height:165px;overflow:hidden;'>" + make_circuit_svg(filter_type) + "</div>"

    # --------------------------------------------------------
    # INFORMATION MESSAGE
    # --------------------------------------------------------

    if filter_type == 'Band-stop':

        info_message.value = """
        <div style="font-size:14px; line-height:1.5; padding:8px 12px;">
        <b>Note on the group delay:</b>
        At the notch frequency, the magnitude response becomes zero, so the phase is undefined at that frequency.
        Since group delay is defined as the negative derivative of phase with respect to angular frequency,
        a direct numerical differentiation produces an artificial very large spike near the notch.
        To avoid displaying this numerical artifact, group-delay values are omitted in a very small neighborhood
        where the magnitude response is effectively zero.
        </div>
        """

    else:

        info_message.value = ''

    # --------------------------------------------------------
    # FREQUENCY RANGE
    # --------------------------------------------------------

    omega = np.logspace(np.log10(wc_val / 10.0), np.log10(wc_val * 10.0), 2000)
    jw = 1j * omega

    # --------------------------------------------------------
    # NUMERICAL TRANSFER FUNCTION
    # --------------------------------------------------------

    if filter_type == 'Low-pass':

        H = 1.0 / (L_val*C_val*jw**2 + (L_val/R_val)*jw + 1.0)

    elif filter_type == 'High-pass':

        H = L_val*C_val*jw**2 / (L_val*C_val*jw**2 + (L_val/R_val)*jw + 1.0)

    elif filter_type == 'Band-pass':

        H = L_val*jw / (R_val*L_val*C_val*jw**2 + L_val*jw + R_val)

    elif filter_type == 'Band-stop':

        H = (L_val*C_val*jw**2 + 1.0) / (L_val*C_val*jw**2 + R_val*C_val*jw + 1.0)

    # --------------------------------------------------------
    # MAGNITUDE, PHASE AND GROUP DELAY
    # --------------------------------------------------------

    magnitude_db = 20.0 * np.log10(np.maximum(np.abs(H), 1e-12))
    phase = np.unwrap(np.angle(H))
    phase_deg = np.degrees(phase)
    group_delay = -np.gradient(phase, omega)

    if filter_type == 'Band-stop':
        group_delay[np.abs(H) < 1e-3] = np.nan

    # --------------------------------------------------------
    # NEW FIGURE FOR EACH UPDATE
    # --------------------------------------------------------

    fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.0))

    ax_mag = axes[0]
    ax_phase = axes[1]
    ax_group = axes[2]

    # --------------------------------------------------------
    # MAGNITUDE RESPONSE
    # --------------------------------------------------------

    ax_mag.semilogx(omega, magnitude_db, 'r-', linewidth=1.7)
    ax_mag.axvline(wc_val, color='gray', linestyle='--', linewidth=1)

    ax_mag.set_title('Magnitude Response')
    ax_mag.set_xlabel('Frequency ω (rad/s)')
    ax_mag.set_ylabel('Gain (dB)')
    ax_mag.set_xlim(wc_val / 10.0, wc_val * 10.0)
    ax_mag.grid(True, which='both', linestyle=':', alpha=0.7)

    # --------------------------------------------------------
    # PHASE RESPONSE
    # --------------------------------------------------------

    ax_phase.semilogx(omega, phase_deg, 'r-', linewidth=1.7)
    ax_phase.axvline(wc_val, color='gray', linestyle='--', linewidth=1)

    ax_phase.set_title('Phase Response')
    ax_phase.set_xlabel('Frequency ω (rad/s)')
    ax_phase.set_ylabel('Phase (deg)')
    ax_phase.set_xlim(wc_val / 10.0, wc_val * 10.0)
    ax_phase.grid(True, which='both', linestyle=':', alpha=0.7)

    # --------------------------------------------------------
    # GROUP DELAY
    # --------------------------------------------------------

    ax_group.semilogx(omega, group_delay, 'r-', linewidth=1.7)
    ax_group.axvline(wc_val, color='gray', linestyle='--', linewidth=1)

    ax_group.set_title('Group Delay')
    ax_group.set_xlabel('Frequency ω (rad/s)')
    ax_group.set_ylabel('Group Delay (s)')
    ax_group.set_xlim(wc_val / 10.0, wc_val * 10.0)
    ax_group.grid(True, which='both', linestyle=':', alpha=0.7)

    # --------------------------------------------------------
    # AXIS FORMAT
    # --------------------------------------------------------

    major_ticks = wc_val * np.array([0.1, 1.0, 10.0])
    minor_ticks = wc_val * np.concatenate((np.arange(2, 10) / 10.0, np.arange(2, 10)))

    for ax in axes:

        ax.set_xticks(major_ticks)
        ax.set_xticks(minor_ticks, minor=True)
        ax.xaxis.set_minor_formatter(NullFormatter())

        ax.tick_params(axis='x', labelsize=8)
        ax.tick_params(axis='y', labelsize=8)

        ax.set_frame_on(True)

        for spine in ['left', 'right', 'top', 'bottom']:

            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_linewidth(0.8)
            ax.spines[spine].set_clip_on(False)

    fig.subplots_adjust(left=0.07, right=0.98, bottom=0.22, top=0.86, wspace=0.38)

    plt.show()
    plt.close(fig)

# ------------------------------------------------------------
# INTERACTIVE WIDGET
# ------------------------------------------------------------

widget_plot = interactive(plot_filter, filter_type=filter_radio, R_val=r_slider, L_val=l_slider, C_val=c_slider)

# ------------------------------------------------------------
# OUTPUT OF INTERACTIVE
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

# ------------------------------------------------------------
# REMOVE OUTPUT SCROLL BARS
# ------------------------------------------------------------

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.output_scroll {
    max-height: none !important;
    height: auto !important;
    overflow: visible !important;
    overflow-y: visible !important;
    overflow-x: visible !important;
}

</style>
"""))

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

display(top_row)
display(slider_row)
display(plot_output)
display(info_message)